Import important packages

In [81]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


Loading data and changing missing data values to Null

In [82]:
df = pd.read_csv("data/train.csv")
# Remove ? with NULL
df = df.replace(r"^\s*\?\s*$", pd.NA, regex=True)
# Remove Unkown/Invalid with NULL
df = df.replace(r"^\s*Unknown/Invalid\s*$", pd.NA, regex=True)
# Remove empty values with NULL
df = df.replace(r"^\s*$", pd.NA, regex=True)

#----------------------------------
#Setting unkown values from IDS_mapping to NULL in train.csv

# From IDS_mapping.csv
ids_to_null = {
    "admission_type_id": [5, 6, 8],                 # Not Available, NULL, Not Mapped
    "discharge_disposition_id": [18, 25, 26],       # NULL, Not Mapped, Unknown/Invalid
    "admission_source_id": [9, 15, 17, 20, 21],     # Not Available, NULL, Not Mapped, Unknown/Invalid
}

for col, bad_codes in ids_to_null.items():
    df[col] = df[col].replace(bad_codes, pd.NA)



checking that data is gone

In [83]:
# Check that mapped ID placeholders are gone
print((df["admission_type_id"].isin([5,6,8])).sum())
print((df["discharge_disposition_id"].isin([18,25,26])).sum())
print((df["admission_source_id"].isin([9,15,17,20,21])).sum())

# Check key placeholder strings are gone
for c in ["weight","payer_code","medical_specialty","race","gender","diag_1","diag_2","diag_3"]:
    print(c, (df[c].astype(str).str.strip() == "?").sum(), (df[c].astype(str).str.strip() == "Unknown/Invalid").sum())


0
0
0
weight 0 0
payer_code 0 0
medical_specialty 0 0
race 0 0
gender 0 0
diag_1 0 0
diag_2 0 0
diag_3 0 0


checking information on featre

In [84]:

df.shape
df.head()
df.info()
df.describe()

<class 'pandas.DataFrame'>
RangeIndex: 71236 entries, 0 to 71235
Data columns (total 51 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   id                        71236 non-null  int64 
 1   encounter_id              71236 non-null  int64 
 2   patient_nbr               71236 non-null  int64 
 3   race                      69615 non-null  str   
 4   gender                    71233 non-null  str   
 5   age                       71236 non-null  str   
 6   weight                    2250 non-null   str   
 7   admission_type_id         64013 non-null  object
 8   discharge_disposition_id  67917 non-null  object
 9   admission_source_id       66359 non-null  object
 10  time_in_hospital          71236 non-null  int64 
 11  payer_code                43058 non-null  str   
 12  medical_specialty         36306 non-null  str   
 13  num_lab_procedures        71236 non-null  int64 
 14  num_procedures            71236 n

,id,encounter_id,patient_nbr,time_in_hospital,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,number_diagnoses
count,71236.000000,7.123600e+04,7.123600e+04,71236.000000,71236.000000,71236.000000,71236.000000,71236.000000,71236.000000,71236.000000,71236.000000
mean,50874.217250,1.651772e+08,5.421577e+07,4.394660,43.112612,1.342523,16.016452,0.371104,0.195800,0.635353,7.418103
std,29423.014266,1.027123e+08,3.866893e+07,2.979942,19.658896,1.705587,8.130801,1.269937,0.864571,1.269287,1.930480
min,2.000000,1.573800e+04,1.350000e+02,1.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,1.000000
25%,25319.750000,8.471072e+07,2.340213e+07,2.000000,31.000000,0.000000,10.000000,0.000000,0.000000,0.000000,6.000000
50%,50867.500000,1.523227e+08,4.520375e+07,4.000000,44.000000,1.000000,15.000000,0.000000,0.000000,0.000000,8.000000
75%,76450.250000,2.307948e+08,8.742009e+07,6.000000,57.000000,2.000000,20.000000,0.000000,0.000000,1.000000,9.000000
max,101766.000000,4.438672e+08,1.895026e+08,14.000000,132.000000,6.000000,81.000000,42.000000,63.000000,19.000000,16.000000


Checking different features unique values

In [85]:
# print Missing % in each feature
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print ("Missing % of data in features")
print(missing_pct.head(20))

# print Low-variance (few unique values)
nunique = df.nunique(dropna=False).sort_values()
print(nunique.head(20))

Missing % of data in features
weight                      96.841485
max_glu_serum               94.776517
A1Cresult                   83.323039
medical_specialty           49.034196
payer_code                  39.555843
admission_type_id           10.139536
admission_source_id          6.846258
discharge_disposition_id     4.659161
race                         2.275535
diag_3                       1.388343
diag_2                       0.342523
diag_1                       0.021057
gender                       0.004211
num_lab_procedures           0.000000
age                          0.000000
time_in_hospital             0.000000
patient_nbr                  0.000000
id                           0.000000
encounter_id                 0.000000
number_inpatient             0.000000
dtype: float64
examide                     1
citoglipton                 1
troglitazone                1
acetohexamide               2
tolbutamide                 2
metformin-pioglitazone      2
glimepiride-pio

In [86]:
# Dropping unnecisary features
df = df.drop(columns = ["id",
         "encounter_id",
         "patient_nbr",
         "payer_code",
         "diag_1",
         "diag_2",
         "diag_3"
         ], errors="ignore")

# Dropping features with to many missing values
df = df.drop(columns = [
        "weight",
        "max_glu_serum",
        "A1Cresult",
        "medical_specialty",
        ], errors="ignore")

# Dropping features with no variation
df = df.drop(columns = [
        "troglitazone",
        "examide",
        "citoglipton",
         ], errors="ignore")

removing unsuable features

In [87]:

# print Missing % in each feature
missing_pct = (df.isna().mean() * 100).sort_values(ascending=False)
print ("Missing % of data in features")
print(missing_pct.head(20))

# print Low-variance (few unique values)
nunique = df.nunique(dropna=False).sort_values()
print(nunique.head(20))


Missing % of data in features
admission_type_id           10.139536
admission_source_id          6.846258
discharge_disposition_id     4.659161
race                         2.275535
gender                       0.004211
age                          0.000000
time_in_hospital             0.000000
num_lab_procedures           0.000000
num_procedures               0.000000
num_medications              0.000000
number_outpatient            0.000000
number_emergency             0.000000
number_inpatient             0.000000
number_diagnoses             0.000000
metformin                    0.000000
repaglinide                  0.000000
nateglinide                  0.000000
chlorpropamide               0.000000
glimepiride                  0.000000
acetohexamide                0.000000
dtype: float64
glipizide-metformin         2
glimepiride-pioglitazone    2
tolbutamide                 2
acetohexamide               2
diabetesMed                 2
change                      2
metformin-piogl

In [90]:
# We already cleaned df earlier; now add numeric target
tmp = df.copy()
tmp["readmitted_bin"] = tmp["readmitted"].map({"No": 0, ">30": 1, "<30": 1})

# Evaluate each feature (except target)
features = [c for c in tmp.columns if c not in ["readmitted", "readmitted_bin"]]

for col in features:
    print("\n" + "="*60)
    print(f"Feature: {col}")

    # Numeric: compare group statistics for not-readmitted (0) vs readmitted (1)
    if pd.api.types.is_numeric_dtype(tmp[col]):
        out = tmp.groupby("readmitted_bin")[col].agg(["count", "mean", "median", "std"])
        print(out)

    # Categorical: show readmission rate for each category value
    else:
        out = (
            tmp[[col, "readmitted_bin"]]
            .dropna()
            .groupby(col)["readmitted_bin"]
            .agg(["count", "mean"])
            .rename(columns={"mean": "readmission_rate"})
            .sort_values("readmission_rate", ascending=False)
        )
        print(out.head(15))  # top 15 categories by readmission rate



Feature: race
                 count  readmission_rate
race                                    
Caucasian        53373          0.469114
AfricanAmerican  13351          0.459142
Hispanic          1428          0.426471
Other             1031          0.393792
Asian              432          0.335648

Feature: gender
        count  readmission_rate
gender                         
Female  38235          0.469857
Male    32998          0.450512

Feature: age
          count  readmission_rate
age                              
[70-80)   18179          0.480389
[80-90)   12037          0.478691
[60-70)   15801          0.465793
[20-30)    1165          0.451502
[40-50)    6785          0.445247
[50-60)   12080          0.438742
[30-40)    2650          0.435849
[90-100)   1940          0.395361
[10-20)     495          0.379798
[0-10)      104          0.182692

Feature: admission_type_id
                   count  readmission_rate
admission_type_id                         
1                

In [89]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, mutual_info_classif

y = df["readmitted"].map({"No": 0, ">30": 1, "<30": 1})
X = pd.get_dummies(df.drop(columns=["readmitted"], errors="ignore"), dummy_na=True)

selector = SelectKBest(score_func=mutual_info_classif, k=20)
selector.fit(X, y)

important_features = X.columns[selector.get_support()]
print("Top selected features:")
print(important_features.tolist())


Top selected features:
['number_emergency', 'number_inpatient', 'race_Caucasian', 'discharge_disposition_id_11', 'admission_source_id_7', 'repaglinide_No', 'nateglinide_No', 'chlorpropamide_No', 'glimepiride_No', 'acetohexamide_No', 'glyburide_No', 'tolbutamide_No', 'rosiglitazone_No', 'glyburide-metformin_No', 'glipizide-metformin_No', 'glimepiride-pioglitazone_No', 'metformin-pioglitazone_No', 'change_Ch', 'change_No', 'diabetesMed_Yes']
